# Chapter 3: Galois Group Classification of $K_tK_{\overline{t}} / \mathbb{Q}$

**Context:**
This script provides some computations for the proof of Lemmas 3.1.9 and 3.1.10. These lemmas are used to prove Proposition 3.1.1, which classifies the Galois group $\mathcal{G}_t = \operatorname{Gal}(K_tK_{\overline{t}}/\mathbb{Q})$ depending on the intersections of the fields $K_t$ and $K_{\overline{t}}$.

**Methodology:**
* While most $t$ result in $\mathcal{G}_t \simeq (C_4 \times C_4) \rtimes C_2$ or $(C_4 \times C_2) \rtimes C_2$, (or $D_{4}$ when $\operatorname{Re}(t)=0$) certain specific values of $t$ with $\operatorname{Re}(t)\neq 0$ cause the group to collapse into $D_4$ or $C_4 \times C_2$. 
* **Lemma 3.1.9 ($D_4$ Check):** Bounds real quadratic integers to find candidates for $\gamma\overline{\gamma}$, solving for $t$ to prove $D_4$ occurs only for $t = 3 + \sqrt{-14}$.
* **Lemma 3.1.10 ($C_4 \times C_2$ Check):** Filters coefficients $A, B$ to prove abelian cases only occur for $t = \frac{1}{2} + \frac{5}{2}\sqrt{-3}$ and $\frac{5}{2} + \frac{3}{2}\sqrt{-11}$.

**Key Functions:**
* `search_dihedral_t()`: Searches valid pairs of constants to find candidate values for $t$ that could yield a $D_4$ Galois group.
* `verify_dihedral_galois()`: Computes the Galois group for the specific field generated by $t = 3 + \sqrt{-14}$ to verify the $D_4$ collapse.
* `search_and_verify_abelian()`: Filters out pairs $(A,B)$, obtains values of $t$ that yield the $C_4 \times C_2$ group and verifies them algebraically.

In [1]:
import itertools
from sage.all import *

# =============================================================================
# PART 1: The Dihedral (D4) Search and Verification (Lemma 3.1.9)
# =============================================================================

def find_quadratic_integers():
    """
    Finds all real quadratic integers a + b*sqrt(d) satisfying:
    8 <= 2*a <= 1028, 0 <= 2*b*sqrt(d) <= 1020, and (2a)^2 - d*(2b)^2 = 64.
    Returns a set of tuples (a, b, d).
    """
    real_quad_ints = set()
    
    for two_a in range(8, 1029):
        bd2 = two_a**2 - 64
        
        if bd2 < 0 or Integer(bd2).is_square():
            continue
            
        d = Integer(bd2).squarefree_part()
        two_b = ZZ(sqrt(bd2 / d))
        
        a = QQ(two_a) / 2
        b = QQ(two_b) / 2
        
        gamma_norm = a + b * sqrt(d)
        if bool(abs(gamma_norm) < 4) or bool(abs(gamma_norm) > 64):
            continue
            
        real_quad_ints.add((a, b, d))
        
    return real_quad_ints

def find_A1_B1_candidates():
    """
    Given the valid real quadratic integers, finds all valid pairs (A1, B1) 
    that lead to integer values and satisfy the bounds in the proof.
    """
    ggbars = find_quadratic_integers()
    candidates = set()
    
    for a, b, d in ggbars:
        L2 = NumberField(x**2 - d, 'rootd')
        rootd = L2.gen()
        gamma_gammabar = a + b * rootd
        
        for A1 in [1, 2, 3]:
            B1 = 4 * A1**2 * gamma_gammabar / (gamma_gammabar - 4)**2
            if B1 in ZZ and (4 * A1 + B1) <= 16:
                candidates.add(((a, b, d), A1, B1))
                
    return candidates

def search_dihedral_t():
    """
    Solves for t using the candidates for gamma*gamma_bar, A1, and B1.
    Returns the valid values of t leading to the D4 Galois group.
    """
    candidates = find_A1_B1_candidates()
    valid_t_results = []
    
    for (a, b, d), A1, B1 in candidates:
        trace_g = 2 * a
        A1, B1 = ZZ(A1), ZZ(B1)
        
        cos2g = 4 * A1 + B1  
        sin2g = 16 - cos2g   
        
        tcos2tp = trace_g * cos2g / 4 + 2 * cos2g
        tcos2tm = trace_g * cos2g / 4 - 2 * cos2g
        
        tsin2tp = -trace_g * sin2g / 4 - 32 - 2 * cos2g
        tsin2tm = -trace_g * sin2g / 4 - 2 * sin2g
        
        dvalp = tsin2tp.squarefree_part()
        dvalm = tsin2tm.squarefree_part()
        
        tsin2tpadj = sqrt(tsin2tp / dvalp) / 2
        tsin2tmadj = sqrt(tsin2tm / dvalm) / 2
        
        ttbarp = cos2g + trace_g + 8
        ttbarm = -cos2g + trace_g + 8
        
        mp = ttbarp**2 + 16 * (tcos2tp - 2 * ttbarp) + 256
        mm = ttbarm**2 + 16 * (tcos2tm - 2 * ttbarm) + 256
        
        if tcos2tm in ZZ and Integer(tcos2tm).is_square() and Integer(mm).is_square():
            valid_t_results.append({
                't_string': f"+/-{sqrt(tcos2tm / 4)} +/- {tsin2tmadj}*sqrt({dvalm})",
                'context': f"A1={A1}, B1={B1}"
            })
            
    return valid_t_results

def verify_dihedral_galois():
    """Verifies that t = 3 + sqrt(-14) yields Galois Group D4."""
    Lt = NumberField(x**2 - 6*x + 23, 't')
    t = Lt.gen()
    Ry = PolynomialRing(Lt, 'y')
    y = Ry.gen()
    
    Kt = NumberField(y**4 - t*y**3 - 6*y**2 + t*y + 1, 'alpha')
    return Kt.galois_group()


# =============================================================================
# PART 2: The Abelian (C4 x C2) Search and Verification (Lemma 3.1.10)
# =============================================================================

def t_plus_tbar_squared(A, B):
    """Computes the value of (t + tbar)^2 coming from a given pair (A,B)."""
    denom = 16 - 4 * A + B
    if denom == 0:
        return False
    return B * (A**2 - 4 * B) / denom

def find_abelian_ab_pairs():
    """
    Filters out pairs (A,B) that do not have the properties outlined in the proof.
    Returns a set of valid (A, B) pairs.
    """
    ab_pairs = set()
    
    for A, B in itertools.product(range(2, 8), range(1, 13)):
        discriminant = A**2 - 4 * B
        
        if discriminant < 0:
            continue
            
        if discriminant > 0 and sqrt(discriminant) in QQ:
            continue
            
        if 16 - 4 * A + B <= 0:
            continue
            
        x_coef_sq = t_plus_tbar_squared(A, B)
        
        if x_coef_sq is False or x_coef_sq not in ZZ or not Integer(x_coef_sq).is_square():
            continue
            
        ab_pairs.add((A, B))
        
    return ab_pairs

def search_and_verify_abelian():
    """
    Takes valid pairs, computes t (if Re(t) != 0), and verifies 
    that it leads to the Galois group C4 x C2.
    """
    ab_pairs = find_abelian_ab_pairs()
    results = []
    
    for A, B in ab_pairs:
        rt_sq = t_plus_tbar_squared(A, B)
        
        if rt_sq == 0:
            continue
            
        ttbar = (16 * rt_sq + 64 * B - 4 * B**2) / (4 * B)
        mdbsq = rt_sq - 4 * ttbar  
        
        d_val = mdbsq.squarefree_part()
        
        Lt = NumberField(x**2 - sqrt(rt_sq)*x + ttbar, 't')
        t = Lt.gen()
        Ry = PolynomialRing(Lt, 'y')
        y = Ry.gen()
        Kt = NumberField(y**4 - t*y**3 - 6*y**2 + t*y + 1, 'alpha')
        
        results.append({
            't_string': f"+/-{sqrt(rt_sq)/2} +/- {sqrt(mdbsq/d_val)/2}*sqrt({d_val})",
            'context': f"A={A}, B={B}",
            'galois_grp': Kt.galois_group()
        })
        
    return results


# =============================================================================
# MAIN RUNNER
# =============================================================================

if __name__ == "__main__":
    print("=== Lemma 3.1.9: Dihedral (D4) Search and Verification ===")
    dihedral_results = search_dihedral_t()
    d4_group = verify_dihedral_galois()
    
    for res in dihedral_results:
        print(f"Found t candidate (from {res['context']}): t = {res['t_string']}")
        print(f"   -> Verification: K_t has Galois Group: {d4_group}")
        
    print("\n=== Lemma 3.1.10: Abelian (C4 x C2) Search and Verification ===")
    abelian_results = search_and_verify_abelian()
    
    for res in abelian_results:
        print(f"Found t candidate (from {res['context']}): t = {res['t_string']}")
        print(f"   -> Verification: K_t has Galois Group: {res['galois_grp']}")

=== Lemma 3.1.9: Dihedral (D4) Search and Verification ===
Found t candidate (from A1=2, B1=1): t = +/-3 +/- 1*sqrt(-14)
   -> Verification: K_t has Galois Group: Galois group 8T4 ([4]2) with order 8 of y^4 - t*y^3 - 6*y^2 + t*y + 1

=== Lemma 3.1.10: Abelian (C4 x C2) Search and Verification ===
Found t candidate (from A=5, B=5): t = +/-5/2 +/- 3/2*sqrt(-11)
   -> Verification: K_t has Galois Group: Galois group 8T2 (4[x]2) with order 8 of y^4 - t*y^3 - 6*y^2 + t*y + 1
Found t candidate (from A=3, B=1): t = +/-1/2 +/- 5/2*sqrt(-3)
   -> Verification: K_t has Galois Group: Galois group 8T2 (4[x]2) with order 8 of y^4 - t*y^3 - 6*y^2 + t*y + 1
